# Analyse des doublons 
Bronze => trop laborieux 
Silver ?

In [ ]:
import sys
print(sys.executable)

In [ ]:
!{sys.executable} -m pip install scikit-learn

In [ ]:
import matplotlib.pyplot as plt
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Accessing Storage Files: Bronze → Silver
Explore how to access and manipulate job files stored in the bronze and silver layers using `get_storage_from_env()` and the project environment configuration.

In [ ]:
# Map du python path dans le docker
#import sys, os
#sys.path.insert(0, os.path.abspath('../..'))  # remonte à la racine du projet
#from src.config.env import load_project_env

## Load Project Environment
Import and load the project environment configuration to initialize environment variables.

In [ ]:
#AS
import os
from pathlib import Path

print("Working dir:", os.getcwd())
print("Notebook file parent (approx):", Path().resolve())
print("Listing here:", sorted(p.name for p in Path().resolve().iterdir() if not p.name.startswith('.'))[:20])

In [ ]:
#AS
import sys
sys.path.insert(0, "../../..")  # <-- mets ici le dossier qui contient `src`

In [ ]:
# Deactivate warning
import warnings
warnings.filterwarnings('ignore')

# Load project environment
from src.config.env import load_project_env
load_project_env()  # Safe to call multiple times (idempotent)
print("✅ Project environment loaded successfully")

## Create storage connections for silver layers using `get_storage_from_env()` for the 'welcometothejungle' source.

In [ ]:
from src.storage.storage import get_storage_from_env
import src.utils.merge_dataset_utils as merge_utils
#import logging
#logger = logging.getLogger(__name__)

storage_wttj = get_storage_from_env("silver", "merged")

## Load wttj parquet file with helper

In [ ]:
df = merge_utils.read_wttj_parquet_file_to_df(storage_wttj,"")

In [ ]:
df.head(5)

## Stats 

In [ ]:
df.info()

In [ ]:
df['source'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%'

In [ ]:
# 1. Nb offres par source
print("\n1. Nb offres par source\n", df.groupby('source')['id'].count())

# 2. Nb entreprises uniques par source
print("\n2. Nb entreprises uniques par source\n", df.groupby('source')['company_name'].nunique())

# 3. Entreprises présentes sur les deux plateformes
print("\n3. Entreprises présentes sur les deux plateformes\n")
ent_ft   = set(df[df['source']=='FT']['company_name'].str.lower().str.strip())
ent_wttj = set(df[df['source']=='WTTJ']['company_name'].str.lower().str.strip())
print(f"Entreprises sur FT seulement     : {len(ent_ft - ent_wttj)}")
print(f"Entreprises sur WTTJ seulement   : {len(ent_wttj - ent_ft)}")
print(f"Entreprises sur les deux         : {len(ent_ft & ent_wttj)}")

# 4. Distribution temporelle
print("\n4. Distribution temporelle\n", df.groupby('source')['published_at'].agg(['min', 'max']))

In [ ]:

# Regarder ce qu'il y a dans company_name pour WTTJ
print(df[df['source']=='WTTJ']['company_name'].value_counts().head(10))

In [ ]:
print(df[df['source']=='WTTJ'][['title', 'description', 'company_url', 'profile']].head(3).to_string())


In [ ]:
# company_url donne peut-être le slug de l'entreprise
print(df[df['source']=='WTTJ']['company_url'].value_counts().head(10))


## Doublons ?
- il faut 'source' de valeurs différentes
- après normalisation de qqch ? 
- sur quels critères on teste

In [ ]:
# Nb et % par source
source_counts = df['source'].value_counts()
source_pct = (df['source'].value_counts(normalize=True) * 100).round(1)

pd.DataFrame({
    'nb_offres': source_counts,
    'pct': source_pct.astype(str) + '%'
})

In [ ]:
cols = ['title', 'company_name', 'job_city', 'published_at']

print("=== France Travail ===")
df[df['source'] == 'FT'][cols].sort_values('title').head(10)

In [ ]:
print("=== WTTJ ===")
df[df['source'] == 'WTTJ'][cols].sort_values('title').head(10)

In [ ]:
#est ce que je peux trouver un même titre approx
mask = df['title'].str.contains('adjoint au chef', case=False, na=False)
df[mask][['title', 'source']].sort_values('title')

In [ ]:
df[mask].groupby('source').size()

### Titres en commun

In [ ]:
titres_ft = set(df[df['source'] == 'FT']['title'].str.lower().str.strip())
titres_wttj = set(df[df['source'] == 'WTTJ']['title'].str.lower().str.strip())

titres_communs = titres_ft & titres_wttj
print(f"{len(titres_communs)} titres en commun\n")

In [ ]:
#display(titres_communs)

In [ ]:
df[df['title'].str.lower().str.strip().isin(titres_communs)][['title', 'source']].sort_values('title')


### Tentative matching flou sur le titre

In [ ]:
titles_ft   = df[df['source'] == 'FT'][['title', 'title_norm']].drop_duplicates()
titles_wttj = df[df['source'] == 'WTTJ'][['title', 'title_norm']].drop_duplicates()

In [ ]:
print(titles_ft.shape[0], "titres FT")
print(titles_wttj.shape[0], "titres WTTJ")

In [ ]:

list_ft   = titles_ft['title_norm'].tolist()
list_wttj = titles_wttj['title_norm'].tolist()

# Vectorisation
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,4))
all_titles = list_ft + list_wttj
tfidf_matrix = vectorizer.fit_transform(all_titles)

matrix_ft   = tfidf_matrix[:len(list_ft)]
matrix_wttj = tfidf_matrix[len(list_ft):]

# Matching par batch pour éviter les OOM
batch_size = 1000
results = []

for i in range(0, len(list_ft), batch_size):
    batch = matrix_ft[i:i+batch_size]
    scores = cosine_similarity(batch, matrix_wttj)
    best_idx = np.argmax(scores, axis=1)
    best_score = scores[np.arange(len(best_idx)), best_idx]
    
    for j, (idx, score) in enumerate(zip(best_idx, best_score)):
        if score >= 0.8:
            results.append({
                'title_ft': list_ft[i+j],
                'title_wttj': list_wttj[idx],
                'score': score
            })

df_matches = pd.DataFrame(results)
print(f"{len(df_matches)} paires trouvées")


In [ ]:
df_matches.sort_values('score', ascending=False).head(10)


In [ ]:
#nb de match par tranche de score
pd.cut(df_matches['score'], bins=[0.75, 0.8,
                                  0.85, 0.9,
                                  0.95, 1.0, 1.001]).value_counts().sort_index(ascending=False)

### Tentative matching flou sur le titre/entreprise name/job_city/description 
On va nettoyer un peu les champs titre, company_name, job_city, description : lower case, supprimer les ponctuations, ...

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text

df['title_clean'] = df['title'].apply(clean_text)
df['company_name_clean'] = df['company_name'].apply(clean_text)
df['job_city_clean'] = df['job_city'].apply(clean_text)
df['description_clean'] = df['description'].apply(clean_text)


In [ ]:
df['text_combined'] = (
    df['title_clean'] + ' ' +
    df['company_name_clean'] + ' ' +
    df['job_city_clean'] + ' ' +
    df['description_clean']
)

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['text_combined'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Récupérer les paires avec score > seuil
threshold = 0.8
matches = []

for i in range(len(cosine_sim)):
    for j in range(i+1, len(cosine_sim)):
        if cosine_sim[i][j] >= threshold:
            matches.append({
                'idx1': i,
                'idx2': j,
                'score': cosine_sim[i][j]
            })

matches_df = pd.DataFrame(matches)
print(f"Doublons détectés : {len(matches_df)}")


## Contracts

In [ ]:
# contract_counts = df['contract_type'].value_counts()
contract_counts_by_source = df.groupby('source')['contract_type'].value_counts()

In [ ]:
display(df.loc[(df['source']=='WTTJ') & df['title'].notna()].sort_values('title')[['title','source']].head(150))

### Avant normalisation

In [ ]:
display(df_normalize_contract_counts_by_source.head(30))
#print(f" Libellé unique = {len(contract_counts_by_source)}")


### Après normalisation 

In [ ]:
def normalize_contracts(df, patterns):
    """
    Normalise les types de contrat :
    - Extrait le type principal (CDI, CDD, Intérim, etc.)
    - Extrait le détail (durée, précision)
    - Stocke dans contract_normalized et contract_detail
    """
    def extract_type(value):
        if pd.isna(value):
            return 'Inconnu'
        for pattern, label in patterns:
            if re.search(pattern, str(value)):
                return label
        return str(value)  # garder la valeur originale si pas de match

    def extract_detail(value):
        if pd.isna(value):
            return None
        # Supprimer le pattern trouvé et retourner le reste
        remaining = str(value)
        for pattern, label in patterns:
            remaining = re.sub(pattern, '', remaining).strip()
        # Nettoyer les séparateurs résiduels (-, :, espaces)
        remaining = re.sub(r'^[\s\-:]+|[\s\-:]+$', '', remaining)
        return remaining if remaining else None
    
    df = df.copy()
    df['contract_normalized'] = df['contract_type'].apply(extract_type)
    df['contract_detail']     = df['contract_type'].apply(extract_detail)
    
    return df


In [ ]:
def plot_contract_by_source(df, source, contract_column):
    # Filtrer par source
    df_source = df[df['source'] == source]
    
    # Calcul en pourcentage
    contract_pct = df_source[contract_column].value_counts(normalize=True).mul(100).round(1)
    
    # Graphique
    fig, ax = plt.subplots(figsize=(10, 5))
    contract_pct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    
    # Afficher les % sur les barres
    for i, v in enumerate(contract_pct):
        ax.text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')
    
    ax.set_title(f"Répartition des types de contrat — {source}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Type de contrat")
    ax.set_ylabel("Pourcentage (%)")
    ax.set_ylim(0, contract_pct.max() + 10)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# Mapping des patterns => type normalisé
patterns = [
    (r'(?i)cdi',                'CDI'),
    (r'(?i)contrat à durée indéterminée',                'CDD'),
    (r'(?i)cdd',                'CDD'),
    (r'(?i)contrat à durée déterminée',                'CDD'),
    (r'(?i)profession\s+lib',   'Profession libérale'),
    (r'(?i)intér?im',           'Intérim'),
    (r'(?i)saisonnier',           'Saisonnier'),
    (r'(?i)profession commerciale',     'Profession commerciale'),
    (r'(?i)franchise',     'Franchise')    
]

# Apply
df_normalize = normalize_contracts(df, patterns)

df_normalize_contract_counts_by_source = df_normalize.groupby('source')['contract_normalized'].value_counts()
print(f"Modalités de contrats FT = {len(df_normalize_contract_counts_by_source.loc['FT'])}")
print(f"Modalités de contrats WTTJ = {len(df_normalize_contract_counts_by_source.loc['WTTJ'])}")

display(df_normalize_contract_counts_by_source.head(30))

plot_contract_by_source(df_normalize, 'FT', 'contract_normalized')
plot_contract_by_source(df_normalize, 'WTTJ', 'contract_normalized')


### Expérience

In [ ]:
experience_level_counts_by_source = df.groupby('source')['experience_level'].value_counts()
experience_description_counts_by_source = df.groupby('source')['experience_description'].value_counts()

### Avant normalisation

**FT**
- experienceExige => D : débutant accepté, E : l’expérience est exigée, S : l’expérience est souhaitée
- experienceLibelle => Libellé de l’expérience ex : Débutant accepté / 1 ans ... 
- experienceCommentaire => Commentaire sur l’expérience. Ex: Expérience dans la vente souhaitée

On a pas la bonne correspondance de colonne.
Il faut prendre `experience_level` pour `wttj` et `experience_description` pour FT


In [ ]:
display(experience_level_counts_by_source.head(30))


In [ ]:
display(experience_description_counts_by_source.head(30))

In [ ]:
# Définition des tranches d'expérience avec leurs indices
EXPERIENCE_LEVELS = [
    (0, 'Débutant',    [r'(?i)débutant', r'(?i)0 an', r'(?i)sans expérience']),
    (1, '0-1 an',      [r'(?i)^1 an', r'(?i)^6 mois', r'(?i)^1 mois', r'(?i)^3 mois', r'(?i)less_than_6_months', r'(?i)6_months_to_1_year']),
    (2, '1-2 ans',     [r'(?i)^2 an', r'(?i)1_to_2_years']),
    (3, '2-3 ans',     [r'(?i)^3 an', r'(?i)^24 mois', r'(?i)2_to_3_years' ]),
    (4, '3-5 ans',     [r'(?i)^4 an', r'(?i)^5 an', r'(?i)4_to_5_years', r'(?i)3_to_4_years']),
    (5, '5-10 ans',    [r'(?i)^6 an', r'(?i)^7 an', r'(?i)^8 an', r'(?i)^9 an', r'(?i)^10 an', r'(?i)5_to_7_years', r'(?i)7_to_10_years']),
    (6, '10+ ans',     [r'(?i)^1[1-9] an', r'(?i)^[2-9][0-9] an', r'(?i)10_to_15_years', r'(?i)more_than_15_years']),
    (-1, 'Non précisé', [r'(?i)expérience exigée', r'(?i)expérience souhaitée']),
   
]

def normalize_experience(df, experience_col):
    """
    Normalise les niveaux d'expérience :
    - experience_normalized : label lisible (ex: '0-1 an')
    - experience_index      : indice numérique (ex: 1) pour trier/comparer
    - experience_detail     : valeur originale nettoyée
    """

    def extract_experience(value):
        if pd.isna(value):
            #To DEBUG
            return -1, 'NAN', None
            #return -1, 'Non précisé', None
        
        str_value = str(value).strip()
        
        for index, label, patterns in EXPERIENCE_LEVELS:
            for pattern in patterns:
                if re.search(pattern, str_value):
                    # Détail = valeur originale
                    return index, label, str_value
        
        #return -1, 'Non précisé', str_value
        # To debug
        return -1, value, str_value
    

    results = df[experience_col].apply(extract_experience)
    
    df = df.copy()
    df['experience_index']      = results.apply(lambda x: x[0])
    df['experience_normalized'] = results.apply(lambda x: x[1])
    df['experience_detail']     = results.apply(lambda x: x[2])
    
    return df


In [ ]:
def get_experience_col(row):
    if row['source'] == 'FT':
        return row['experience_description']
    elif row['source'] == 'WTTJ':
        return row['experience_level']
    else:
        return None 

df_normalize['experience_source_composite'] = df.apply(get_experience_col, axis=1)

# Utilisation
df_normalize = normalize_experience(df_normalize,'experience_source_composite')

# Vérification
#df_normalize[['experience_source_composite', 'experience_index', 'experience_normalized', 'experience_detail']].head(20)

display(df_normalize.groupby('source')['experience_normalized'].value_counts())

## ROME

In [ ]:
rome_count = df['rome_code'].value_counts()
print(rome_count)

## Statistics from helper

In [ ]:
if df is not None and not df.empty:
    merge_utils.print_statistics(df)
else    :
    logger.warning("⚠️ Aucune donnée chargée pour les statistiques FT/WTTJ fusionnées")